# 💧 LFM3-Omni - Any Input, Text or Speech Output

This notebook runs the examples from the [Audio Capabilities](https://docs.liquid.ai/lfm/key-concepts/audio-capabilities), [Speech Generation](https://docs.liquid.ai/lfm/key-concepts/speech-generation), [Vision Capabilities](https://docs.liquid.ai/lfm/key-concepts/vision-capabilities), [Tool Calling](https://docs.liquid.ai/lfm/key-concepts/tool-use) documentation with [`LiquidAI/LFM3-Omni-2B`](https://huggingface.co/LiquidAI/LFM3-Omni-2B). The code matches the documentation, so you can run each example end to end.

## Setup

Install Transformers and the packages used by the examples. `librosa` and `soundfile` load and save audio files, and `datasets` loads a sample recording. A GPU is recommended; the model runs in bfloat16.

In [ ]:
!pip install "transformers>=5.20.2" torch torchvision pillow accelerate datasets librosa soundfile torchcodec

Load the model and processor with `trust_remote_code=True`. The processor bundles the audio codec (`processor.audio_tokenizer`), which turns generated speech codes into a 24 kHz waveform.

In [1]:
import torch
import soundfile as sf
from transformers import AutoModelForMultimodalLM, AutoProcessor

model_id = "LiquidAI/LFM3-Omni-2B"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

model = AutoModelForMultimodalLM.from_pretrained(
    model_id, dtype=dtype, device_map=device, trust_remote_code=True
).eval()
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
codec = processor.audio_tokenizer.to(device=device).eval()

Loading weights: 100%|##########| 665/665 [00:01<00:00, 379.71it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|##########| 203/203 [00:00<00:00, 3437.58it/s]


Every task is a chat conversation, and the system prompt describes the task. Answers are in text by default. Some examples first record a spoken input with [text-to-speech](https://docs.liquid.ai/lfm/key-concepts/speech-generation#text-to-speech). Where the code differs between audio and omni models, the examples have a tab for each.

## Speech recognition

Use the system prompt `Perform ASR in <language>.` with the English name of the language. Pass audio in the user turn as `{"type": "audio", "audio": ...}`, where the value is a file path or URL. The processor resamples it to 16 kHz mono.

This example transcribes a LibriSpeech clip:

In [2]:
import io
from datasets import Audio as AudioFeature, load_dataset

librispeech = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
librispeech = librispeech.cast_column("audio", AudioFeature(decode=False))

sample = librispeech[0]
speech, sr = sf.read(io.BytesIO(sample["audio"]["bytes"]))
sf.write("librispeech.wav", speech, sr)
print("Reference:", sample["text"])

messages = [
    {"role": "system", "content": "Perform ASR in English."},
    {"role": "user", "content": [{"type": "audio", "audio": "librispeech.wav"}]},
]

# Apply chat template
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=512, generation_mode="text")

transcript = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Transcript:", transcript)

Reference: MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL
Transcript: Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel.


In [3]:
from IPython.display import Audio, display

display(Audio("librispeech.wav"))

## Speech translation

Models with speech translation support can translate speech into text. Name the source and target languages in the system prompt, for example `Convert the speech in {source} to written {target}.` or `Translate the following speech from {source} to {target} in text form.` Try the phrasing that works best for your data.

For text-to-text translation, use a [text model](https://docs.liquid.ai/lfm/models/text-models) instead.

This example first records a French sentence with text-to-speech:

In [4]:
tts_messages = [
    {"role": "system", "content": "Perform TTS in French using a male voice."},
    {"role": "user", "content": "On pourrait inventer une nouvelle recette ce soir. Si le résultat est étrange, on lui donnera un nom très chic."},
]

In [5]:
# Apply chat template
inputs = processor.apply_chat_template(
    tts_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

torch.manual_seed(0)
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        generation_mode="audio",
        audio_temperature=0.4,
        audio_repetition_penalty=1.2,
        audio_repetition_window=64,
    )

codes = outputs.audio_codes[:1, :, : outputs.audio_lengths[0]]
with torch.inference_mode():
    speech = codec.decode(codes).audio_values[0].float().cpu().numpy().squeeze()
sf.write("tts_fr.wav", speech, 24_000)

In [6]:
from IPython.display import Audio, display

display(Audio("tts_fr.wav"))

Then translate the recording into English:

In [7]:
messages = [
    {"role": "system", "content": "Convert the speech in French to written English."},
    {"role": "user", "content": [{"type": "audio", "audio": "tts_fr.wav"}]},
]

# Apply chat template
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=512, generation_mode="text")

translation = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Translation:", translation)

Translation: We could invent a new recipe this evening. If the result is strange, we will give it a very cheeky name.


## Speech-to-text chat

In this mode, the model behaves like a text LLM that also accepts speech. No system prompt is needed.

This example first records a spoken question with text-to-speech, then asks it:

In [8]:
tts_messages = [
    {"role": "system", "content": "Perform TTS in American English using a male voice."},
    {"role": "user", "content": "I want to move more, but I have not found an activity I enjoy. How can I try a few options and build a small habit?"},
]
messages = [
    {"role": "user", "content": [{"type": "audio", "audio": "question.wav"}]},
]

Record the question:

In [9]:
# Apply chat template
inputs = processor.apply_chat_template(
    tts_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

torch.manual_seed(0)
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        generation_mode="audio",
        audio_temperature=0.4,
        audio_repetition_penalty=1.2,
        audio_repetition_window=64,
    )

codes = outputs.audio_codes[:1, :, : outputs.audio_lengths[0]]
with torch.inference_mode():
    speech = codec.decode(codes).audio_values[0].float().cpu().numpy().squeeze()
sf.write("question.wav", speech, 24_000)

In [10]:
from IPython.display import Audio, display

display(Audio("question.wav"))

Then ask it:

In [11]:
# Apply chat template
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=1024, generation_mode="text")

answer = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Answer:", answer)

Answer: It is completely normal to feel stuck when you want to move more but can't find an activity you enjoy. The key to building a small habit is to stop looking for a "perfect" activity and start looking for a "low-friction" one. 

Here is a step-by-step guide to trying a few options and building a small, sustainable habit:

### 1. Define "Moving" in a Small, Concrete Way
Before you try anything, define what "moving" means to you. You don't have to move a house or travel to a new city. You can start with micro-movements:
*   **Walking:** Just 10 minutes a day.
*   **Stretching:** 5 minutes in the morning.
*   **Carrying a bag:** Carry a grocery bag to the store.
*   **Walking your dog:** 10 minutes a day.
*   **Cooking a new recipe:** 15 minutes a day.

### 2. Try 3-5 Low-Friction Activities
Pick 3 to 5 activities that feel easy to start, even if they aren't exciting. The goal is to lower the barrier to entry. 
*   *Example:* Walking your dog, carrying a grocery bag, doing 5 minutes

## Audio understanding

Text and audio can be mixed in the same user turn, for example a recording plus a written question about it. Questions can address speech, music, background sounds, or events.

This example asks a question about the LibriSpeech clip from [Speech recognition](https://docs.liquid.ai/lfm/key-concepts/audio-capabilities#speech-recognition):

In [12]:
messages = [{
    "role": "user",
    "content": [
        {"type": "audio", "audio": "librispeech.wav"},
        {"type": "text", "text": "Who or what is the speaker talking about, and what is their attitude towards it?"},
    ],
}]

In [13]:
# Apply chat template
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=512, generation_mode="text")

answer = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Answer:", answer.strip())

Answer: The speaker is talking about Mr. Quilter, whom he describes as the "apostle of the middle classes." His attitude is one of warm welcome and gladness, as he explicitly states, "And we're glad to welcome his gospel."


For multiple choice, use the system prompt `Answer questions about the provided audio clip. Base every answer only on what can be heard.` and end the user turn with `Please choose the answer from the options.`

In [14]:
messages = [
    {"role": "system", "content": "Answer questions about the provided audio clip. Base every answer only on what can be heard."},
    {"role": "user", "content": [
        {"type": "audio", "audio": "librispeech.wav"},
        {"type": "text", "text": "What does the speaker welcome?\nA. A new law\nB. Mr. Quilter's gospel\nC. A visiting choir\nD. The end of the middle classes\nPlease choose the answer from the options."},
    ]},
]

# Apply chat template
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=512, generation_mode="text")

answer = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(answer)

Mr. Quilter's gospel


## Choose the output modality

The model chooses its output modality itself, guided by the system prompt. Pass `generation_mode` to `model.generate()` to override that choice: `"text"`, `"audio"` (speech only), or `"interleaved"`. Use `"audio"` for reading text aloud. For open-ended answers, prefer `"interleaved"`.

## Text-to-speech

Use the system prompt `Perform TTS in <language> using a <female|male> voice.` For English, say `American English` or `British English`.

Check the model page for supported languages and voices.

This example speaks a sentence in American English:

In [15]:
messages = [
    {"role": "system", "content": "Perform TTS in American English using a female voice."},
    {"role": "user", "content": "I made a little plan for today: go for a walk, find a new workout playlist, and try a new cake recipe."},
]

Pass `generation_mode="audio"`:

In [16]:
# Apply chat template
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

torch.manual_seed(0)
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        generation_mode="audio",
        audio_temperature=0.4,
        audio_repetition_penalty=1.2,
        audio_repetition_window=64,
    )

codes = outputs.audio_codes[:1, :, : outputs.audio_lengths[0]]
with torch.inference_mode():
    speech = codec.decode(codes).audio_values[0].float().cpu().numpy().squeeze()
sf.write("tts_en.wav", speech, 24_000)

In [17]:
from IPython.display import Audio, display

display(Audio("tts_en.wav"))

To check the result, pass the file to the [speech recognition](https://docs.liquid.ai/lfm/key-concepts/audio-capabilities#speech-recognition) example.

## Spoken chat

Use the system prompt `Respond with interleaved text and audio.` The model answers conversationally in text and speech at the same time, so speech can start playing before the full answer is written. Keep the system prompt exactly as shown, and give any further instructions in the user turn.

This example first records a spoken question with text-to-speech, then asks it and saves the spoken reply to `reply.wav`:

In [18]:
tts_messages = [
    {"role": "system", "content": "Perform TTS in American English using a male voice."},
    {"role": "user", "content": "I want to move more, but I have not found an activity I enjoy. How can I try a few options and build a small habit?"},
]
messages = [
    {"role": "system", "content": "Respond with interleaved text and audio."},
    {"role": "user", "content": [{"type": "audio", "audio": "question.wav"}]},
]

Record the question:

In [19]:
# Apply chat template
inputs = processor.apply_chat_template(
    tts_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

torch.manual_seed(0)
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        generation_mode="audio",
        audio_temperature=0.4,
        audio_repetition_penalty=1.2,
        audio_repetition_window=64,
    )

codes = outputs.audio_codes[:1, :, : outputs.audio_lengths[0]]
with torch.inference_mode():
    speech = codec.decode(codes).audio_values[0].float().cpu().numpy().squeeze()
sf.write("question.wav", speech, 24_000)

In [20]:
from IPython.display import Audio, display

display(Audio("question.wav"))

Then ask it. Pass `generation_mode="interleaved"`:

In [21]:
# Apply chat template
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        generation_mode="interleaved",
        audio_temperature=0.4,
        audio_repetition_penalty=1.2,
        audio_repetition_window=64,
    )

reply = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Text:", reply)

codes = outputs.audio_codes[:1, :, : outputs.audio_lengths[0]]
with torch.inference_mode():
    speech = codec.decode(codes).audio_values[0].float().cpu().numpy().squeeze()
sf.write("reply.wav", speech, 24_000)

Text: Gotcha, trying a couple of things is a great way to narrow it down. You could pick two or three activities that sound interesting, set a tiny goal like fifteen minutes each day, and keep a quick note of how you feel afterward. Then after a week compare which ones gave you the most enjoyment and make that one a regular habit. Does that sound doable for you?


In [22]:
from IPython.display import Audio, display

display(Audio("reply.wav"))

## Answer in speech

An omni model can answer a question about an image in speech. Add an output instruction after the task prompt, for example `Respond with interleaved text and audio.` This example asks in text about a sample photo:

In [23]:
from transformers.image_utils import load_image

load_image("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg").save("bee.jpg")

messages = [
    {"role": "system", "content": "Answer questions about the provided image. Base every answer only on what can be seen. Respond with interleaved text and audio."},
    {"role": "user", "content": [{"type": "image", "image": "bee.jpg"}, {"type": "text", "text": "Is this insect useful for the garden? Why?"}]},
]

# Apply chat template
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        audio_temperature=0.4,
        audio_repetition_penalty=1.2,
        audio_repetition_window=64,
    )

answer = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(answer)

codes = outputs.audio_codes[:1, :, : outputs.audio_lengths[0]]
with torch.inference_mode():
    speech = codec.decode(codes).audio_values[0].float().cpu().numpy().squeeze()
sf.write("bee_answer.wav", speech, 24_000)

Yes, this insect is useful for the garden because it is a bee, and bees are important pollinators. They help plants like the cosmos in the image reproduce by moving pollen from one flower to another, which allows them to make seeds and fruit.


In [24]:
from IPython.display import Audio, display

display(Audio("bee_answer.wav"))

## Stream speech

For streaming applications, `model.generate()` accepts `streamer` for text tokens and `audio_streamer` for speech frames. The codec decodes those frames incrementally, so speech can be played as it is generated.

## Ask about an image by voice

An omni model can understand images and speech in the same turn, so the question can be a recording instead of text.

Recommended system prompts: `Describe the provided image.` for captions, and `Answer questions about the provided image. Base every answer only on what can be seen.` for questions. When the answer should be in a different modality from the question, add an output instruction after the task prompt, for example `Respond with text.`

This example first records a spoken question with [text-to-speech](https://docs.liquid.ai/lfm/key-concepts/speech-generation#text-to-speech), then asks it about a photo:

In [25]:
tts_messages = [
    {"role": "system", "content": "Perform TTS in American English using a female voice."},
    {"role": "user", "content": "What is this tiny garden visitor doing?"},
]

# Apply chat template
inputs = processor.apply_chat_template(
    tts_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

torch.manual_seed(0)
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        generation_mode="audio",
        audio_temperature=0.4,
        audio_repetition_penalty=1.2,
        audio_repetition_window=64,
    )

codes = outputs.audio_codes[:1, :, : outputs.audio_lengths[0]]
with torch.inference_mode():
    speech = codec.decode(codes).audio_values[0].float().cpu().numpy().squeeze()
sf.write("question.wav", speech, 24_000)

In [26]:
from IPython.display import Audio, display

display(Audio("question.wav"))

![Input photo](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg)

In [27]:
from transformers.image_utils import load_image

load_image("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg").save("bee.jpg")

messages = [
    {"role": "system", "content": "Answer questions about the provided image. Base every answer only on what can be seen. Respond with text."},
    {"role": "user", "content": [{"type": "image", "image": "bee.jpg"}, {"type": "audio", "audio": "question.wav"}]},
]

# Apply chat template
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=512, generation_mode="text")

answer = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(answer)

The tiny garden visitor is a **bee** (a bumblebee, identifiable by its black body with yellow-orange bands and fuzzy white abdomen) perched on the center of the pink cosmos flower. It is **feeding on the flower's nectar** — its head is positioned directly over the orange, pollen‑bearing center of the bloom, with its antennae extended toward the stamens as it collects nectar. In short, the bee is visiting the flower to feed on its nectar.


To get a spoken answer instead, see [Speech Generation](https://docs.liquid.ai/lfm/key-concepts/speech-generation#answer-in-speech).

## Video understanding

Video-capable models can answer questions using a clip's frames. Check the model page for the default frame rate and frame limits. Install `torchcodec` and FFmpeg for video decoding in addition to the packages in [Setup](https://docs.liquid.ai/lfm/key-concepts/vision-capabilities#setup):

In [ ]:
%pip install -q torchcodec

### Describe a video

Pass a local path with `{"type": "video", "video": ...}`. This example downloads a short [sample clip](https://huggingface.co/datasets/raushan-testing-hf/videos-test):

<video controls width="480" src="https://huggingface.co/datasets/raushan-testing-hf/videos-test/resolve/main/sample_demo_1.mp4"></video>

In [28]:
from huggingface_hub import hf_hub_download

video_path = hf_hub_download("raushan-testing-hf/videos-test", "sample_demo_1.mp4", repo_type="dataset")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "video", "video": video_path},
            {"type": "text", "text": "Describe what happens in this video."},
        ],
    }
]

# Apply chat template
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=512, generation_mode="text")

answer = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
if not answer.strip():
    raise RuntimeError("No video description was generated.")
print(answer)

The video shows a young toddler with blonde hair, wearing dark-framed glasses, a light blue sleeveless top, and pink pants, sitting on a bed and reading an open paperback book. The bed has a patterned quilt, a dark wooden crib rail in the background, a white garment draped over the bed, and a small framed photo and striped clothing on the right side of the frame.

At the start, the child is focused on the pages, holding the book open with both hands. Around the two-second mark, the toddler lifts one hand to touch and adjust the glasses, then brings both hands up and claps them together in a playful gesture. Immediately after, the child lowers their hands and resumes turning the pages of the book, flipping through the pages with both hands while keeping their attention on the text. The clip ends with the child still reading, hands on the open pages. A "clideo.com" watermark appears in the bottom-right corner throughout the video.


### Use the soundtrack

An omni model can also use a video's soundtrack. Pass `load_audio_from_video=True` and `processor_kwargs={"use_audio_in_video": True}` so the processor interleaves the soundtrack with the frames:

In [29]:
messages = [
    {"role": "system", "content": "Answer questions about the video using what you see and hear."},
    {
        "role": "user",
        "content": [
            {"type": "video", "video": video_path},
            {"type": "text", "text": "What happens in this video, and what can you hear?"},
        ],
    },
]

# Apply chat template
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    load_audio_from_video=True,
    processor_kwargs={"use_audio_in_video": True},
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=256, generation_mode="text")

answer = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(answer)

In this video, a young toddler with blonde hair, wearing dark-framed glasses and a light blue sleeveless top, sits on a bed and reads an open picture book. The child holds the pages with both hands, turns them, and repeatedly looks down at the text, clearly engaged in reading. In the background, a dark wooden crib, a white comforter, and some clothing are visible on the bed, and a "clideo.com" watermark appears in the bottom right corner.

Overlaid on the footage is a baby's voice saying, "Where? I want to look at the... I want to look." The child's speech matches the on-screen text, indicating the toddler is asking where they want to look in the book—likely pointing out a picture or page they'd like to see next. The overall scene captures a sweet, everyday moment of a baby exploring a book.


## Tool calling

Audio and omni models with tool-calling support can take spoken requests and return tool calls. This example turns one spoken request into two function calls: cancel a streaming-platform subscription at the end of the billing period, then subscribe to a podcast platform's Max plan with monthly billing. StreamBox and PodSpace are fictional services, and the example only prints the proposed calls.

This example shows how to define the tools and generate the tool calls. Executing the calls and generating the final answer work the same as in [Text Models](https://docs.liquid.ai/lfm/key-concepts/tool-use#text-models).

The example first records the spoken request with [text-to-speech](https://docs.liquid.ai/lfm/key-concepts/speech-generation#text-to-speech). Define the request, the tools, and the instructions:

In [30]:
import json

tools = [
    {
        "name": "cancel_subscription",
        "description": "Cancel the signed-in user's subscription to a streaming platform at the specified time.",
        "parameters": {
            "type": "object",
            "properties": {
                "platform": {"type": "string", "description": "Name of the streaming platform, for example StreamBox."},
                "effective_when": {
                    "type": "string",
                    "enum": ["immediately", "end_of_billing_period"],
                },
            },
            "required": ["platform", "effective_when"],
        },
    },
    {
        "name": "subscribe_to_podcast_platform",
        "description": "Subscribe the signed-in user to a podcast platform with the chosen plan and billing frequency.",
        "parameters": {
            "type": "object",
            "properties": {
                "platform": {"type": "string", "description": "Name of the podcast platform, for example PodSpace."},
                "plan": {"type": "string", "enum": ["basic", "plus", "max"]},
                "billing_frequency": {"type": "string", "enum": ["monthly", "annual"]},
            },
            "required": ["platform", "plan", "billing_frequency"],
        },
    },
]
instructions = (
    "Return tool calls for both requested actions, in the order requested. "
    "Do not claim that an action has completed without a tool result."
)

tts_messages = [
    {"role": "system", "content": "Perform TTS in British English using a female voice."},
    {"role": "user", "content": "Please cancel my StreamBox subscription at the end of the current billing period. Then subscribe me to PodSpace on the Max plan, billed monthly."},
]

Record the request:

In [31]:
# Apply chat template
inputs = processor.apply_chat_template(
    tts_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

torch.manual_seed(0)
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        generation_mode="audio",
        audio_temperature=0.4,
        audio_repetition_penalty=1.2,
        audio_repetition_window=64,
    )

codes = outputs.audio_codes[:1, :, : outputs.audio_lengths[0]]
with torch.inference_mode():
    speech = codec.decode(codes).audio_values[0].float().cpu().numpy().squeeze()
sf.write("request.wav", speech, 24_000)

In [32]:
from IPython.display import Audio, display

display(Audio("request.wav"))

Pass the tool schemas to `apply_chat_template()` with `tools=...`:

The first response may contain only one call. Return mock results and continue until both requested actions have been validated. The loop below does not change any real subscription.

In [33]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": [{"type": "audio", "audio": "request.wav"}]},
]

# Apply chat template
inputs = processor.apply_chat_template(
    messages,
    tools=tools,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=512, generation_mode="text")

call = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(call)

import re


def parse_tool_calls(text):
    calls = []
    for name, body in re.findall(r"<function=([^>\n]+)>\n(.*?)</function>", text, re.DOTALL):
        arguments = dict(re.findall(r"<parameter=([^>\n]+)>\n(.*?)\n</parameter>", body, re.DOTALL))
        calls.append({"type": "function", "function": {"name": name, "arguments": arguments}})
    return calls


expected = {
    "cancel_subscription": {"platform": "StreamBox", "effective_when": "end_of_billing_period"},
    "subscribe_to_podcast_platform": {"platform": "PodSpace", "plan": "max", "billing_frequency": "monthly"},
}
completed = {}

for _ in range(3):
    calls = parse_tool_calls(call)
    if not calls:
        break
    messages.append({"role": "assistant", "content": "", "tool_calls": calls})
    for tool_call in calls:
        function = tool_call["function"]
        name, arguments = function["name"], function["arguments"]
        if name not in expected or arguments != expected[name]:
            raise ValueError(f"Unexpected subscription call: {function}")
        completed[name] = arguments
        messages.append({"role": "tool", "content": json.dumps({"success": True, "action": name, **arguments})})

    inputs = processor.apply_chat_template(
        messages,
        tools=tools,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=512, generation_mode="text")
    call = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

if completed != expected:
    raise RuntimeError(f"The model did not produce both requested actions: {completed}")
print("Mock actions:", json.dumps(completed))

<tool_call>
<function=cancel_subscription>
<parameter=platform>
StreamBox
</parameter>
<parameter=effective_when>
end_of_billing_period
</parameter>
</function>
</tool_call>
Mock actions: {"cancel_subscription": {"platform": "StreamBox", "effective_when": "end_of_billing_period"}, "subscribe_to_podcast_platform": {"platform": "PodSpace", "plan": "max", "billing_frequency": "monthly"}}


**Image and speech together**

An omni model can also combine what it sees and hears in one tool call. This example shows the model a restaurant photo and asks by voice to book a table. The model reads the restaurant name from the sign and combines it with the city, date, time, party size, and reservation name from the spoken request. The example only prints the proposed call and doesn't make a reservation.

Use an explicit date, written out in the TTS input, so the example is reproducible. Replace both date values together for a different booking. Listen to the generated request; TTS can change details. The final cell validates all six proposed arguments and never makes a reservation.

In [34]:
booking_date = "2026-11-23"
spoken_date = "November twenty-third, twenty twenty-six"
tts_messages = [
    {"role": "system", "content": "Perform TTS in British English using a male voice."},
    {"role": "user", "content": f"Please book a table for two at this restaurant in Paris on {spoken_date} at seven in the evening, under Alex Taylor."},
]

# Apply chat template
inputs = processor.apply_chat_template(
    tts_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

torch.manual_seed(0)
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        generation_mode="audio",
        audio_temperature=0.4,
        audio_repetition_penalty=1.2,
        audio_repetition_window=64,
    )

codes = outputs.audio_codes[:1, :, : outputs.audio_lengths[0]]
with torch.inference_mode():
    speech = codec.decode(codes).audio_values[0].float().cpu().numpy().squeeze()
sf.write("booking.wav", speech, 24_000)

In [35]:
from IPython.display import Audio, display

display(Audio("booking.wav"))

Then show the photo and ask:

The cell reuses `restaurant.jpg` if it is already present. If Wikimedia rate-limits a download, open the credited photo link below in your browser and upload it as `restaurant.jpg`, then rerun the cell. Keep the same photo when checking the expected restaurant name.

In [36]:
import json
import pathlib
import re

from urllib.request import Request, urlopen
from PIL import Image

tools = [{
    "name": "book_table",
    "description": "Book a table at a restaurant using its name and city.",
    "parameters": {
        "type": "object",
        "properties": {
            "restaurant_name": {"type": "string"},
            "city": {"type": "string"},
            "date": {"type": "string", "description": "Local date, YYYY-MM-DD."},
            "time": {"type": "string", "description": "Local time, HH:MM in 24-hour format."},
            "party_size": {"type": "integer", "minimum": 1},
            "reservation_name": {"type": "string"},
        },
        "required": ["restaurant_name", "city", "date", "time", "party_size", "reservation_name"],
    },
}]

photo_url = "https://upload.wikimedia.org/wikipedia/commons/8/80/Cafe_de_Flore_Paris_France.jpg"
if not pathlib.Path("restaurant.jpg").is_file():
    with urlopen(Request(photo_url, headers={"User-Agent": "LFM3Cookbook"}), timeout=30) as response:
        Image.open(response).convert("RGB").save("restaurant.jpg")

messages = [
    {"role": "system", "content": "Use the tools to help the user complete their goal."},
    {"role": "user", "content": [
        {"type": "image", "image": "restaurant.jpg"},
        {"type": "audio", "audio": "booking.wav"},
    ]},
]

# Apply chat template
inputs = processor.apply_chat_template(
    messages,
    tools=tools,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=512, generation_mode="text")

call = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(call)

booking_calls = re.findall(r"<function=([^>\n]+)>\n(.*?)</function>", call, re.DOTALL)
expected_booking = {
    "restaurant_name": "Café de Flore",
    "city": "Paris",
    "date": booking_date,
    "time": "19:00",
    "party_size": "2",
    "reservation_name": "Alex Taylor",
}
if len(booking_calls) != 1 or booking_calls[0][0] != "book_table":
    raise RuntimeError(f"Expected one book_table call, got: {booking_calls}")
arguments = dict(re.findall(r"<parameter=([^>\n]+)>\n(.*?)\n</parameter>", booking_calls[0][1], re.DOTALL))
if arguments != expected_booking:
    raise ValueError(f"Booking arguments do not match the request: {arguments}")
print("Verified booking arguments:", json.dumps(arguments, ensure_ascii=False))

<tool_call>
<function=book_table>
<parameter=restaurant_name>
Café de Flore
</parameter>
<parameter=city>
Paris
</parameter>
<parameter=date>
2026-11-23
</parameter>
<parameter=time>
19:00
</parameter>
<parameter=party_size>
2
</parameter>
<parameter=reservation_name>
Alex Taylor
</parameter>
</function>
</tool_call>
Verified booking arguments: {"restaurant_name": "Café de Flore", "city": "Paris", "date": "2026-11-23", "time": "19:00", "party_size": "2", "reservation_name": "Alex Taylor"}


![Café de Flore, Paris](https://upload.wikimedia.org/wikipedia/commons/8/80/Cafe_de_Flore_Paris_France.jpg)

Photo: [Café de Flore, Paris](https://commons.wikimedia.org/wiki/File:Cafe_de_Flore_Paris_France.jpg) by ayustety, [CC BY-SA 2.0](https://creativecommons.org/licenses/by-sa/2.0/).

# 🚀 Next steps

- Fine-tune LFM3-Omni on your own data: see [`lfm3_omni_vqa_sft_with_transformers.ipynb`](../finetuning/notebooks/lfm3_omni_vqa_sft_with_transformers.ipynb).
- Speech-only workloads: [`lfm3_audio_capabilities.ipynb`](lfm3_audio_capabilities.ipynb) covers LFM3-Audio.
- Documentation: [docs.liquid.ai](https://docs.liquid.ai/lfm) · Community: [Discord](https://discord.com/invite/liquid-ai)